# 04 Orchestrated run (scheduled by the Fabric Data Pipeline)
Runs all stages in order with logging, retries, the quality gate and reconciliation. Raises an error when the run fails, so the pipeline activity turns red and its failure path (alert email) fires.

In [ ]:
# Parameters (the Data Pipeline overrides these)
load_mode = 'full'  # or 'incremental'
raw_path = 'Files/raw'  # e.g. 'Files/increments/2026-11-15' for a change extract

In [ ]:
%pip install https://github.com/alexanderburblis/mfg-ops-intelligence/archive/refs/heads/main.zip --quiet

In [ ]:
# Requires: this notebook attached to the lakehouse 'lh_mfg_ops' (default lakehouse).
# The mfgops package is installed by the %pip cell above (pin a tag or use an Environment in production).
from mfgops.lakehouse import Lakehouse
lh = Lakehouse.detect(spark)
print("Running in Fabric:", lh.fabric)

In [ ]:
from mfgops.orchestrate import run_pipeline
lh.raw_path = raw_path
res = run_pipeline(lh, max_error_rate=0.02, mode=load_mode)
print(res.run_id, res.mode, res.status, res.failed_stage, res.message)

In [ ]:
display(lh.read('meta', 'pipeline_runs').filter(f"run_id = '{res.run_id}'").orderBy('started_at'))

In [ ]:
if res.status != 'success':
    raise RuntimeError(f'Pipeline run {res.run_id} failed at {res.failed_stage}: {res.message}')